In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve, auc
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import mlflow
import mlflow.sklearn
import warnings
warnings.filterwarnings('ignore')

# 1. Quick Data Prep (making this notebook self-contained)
df = pd.read_csv('../data/raw/bank-additional-full.csv', sep=';')
if 'duration' in df.columns: 
    df = df.drop(columns=['duration'])
df['y'] = df['y'].map({'no': 0, 'yes': 1})

X = pd.get_dummies(df.drop(columns=['y']), drop_first=True)
y = df['y']

# Split and apply SMOTE
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# 2. Set up MLflow Tracking
# This creates a local folder called 'mlruns' to store your experiments
mlflow.set_experiment("Bank_Marketing_Campaign")

def train_and_log_model(model, model_name):
    # Start an MLflow run
    with mlflow.start_run(run_name=model_name):
        
        # Train the model
        model.fit(X_train_resampled, y_train_resampled)
        
        # Make predictions
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]
        
        # Calculate Metrics
        roc_auc = roc_auc_score(y_test, y_prob)
        # For imbalanced datasets, PR-AUC is often a better metric than ROC-AUC
        precision, recall, _ = precision_recall_curve(y_test, y_prob)
        pr_auc = auc(recall, precision) 
        
        # Log Parameters, Metrics, and the Model artifact itself to MLflow
        mlflow.log_params(model.get_params())
        mlflow.log_metric("roc_auc", roc_auc)
        mlflow.log_metric("pr_auc", pr_auc)
        mlflow.sklearn.log_model(model, artifact_path=f"models/{model_name}")
        
        print(f"--- {model_name} ---")
        print(f"ROC-AUC:  {roc_auc:.4f}")
        print(f"PR-AUC:   {pr_auc:.4f}")
        print("Status: Logged successfully to MLflow.\n")

# 3. Execute Experiments
print("Starting Model Training Pipeline...\n")

# Run 1: XGBoost
xgb = XGBClassifier(random_state=42, eval_metric='logloss')
train_and_log_model(xgb, "XGBoost_Baseline")

# Run 2: LightGBM (often faster and handles categories well)
lgbm = LGBMClassifier(random_state=42, verbose=-1)
train_and_log_model(lgbm, "LightGBM_Baseline")

/Users/nikaergemlidze/rm-env/lib/python3.11/site-packages/mlflow/utils/requirements_utils.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources  # noqa: TID251
2026/04/18 05:38:47 INFO mlflow.tracking.fluent: Experiment with name 'Bank_Marketing_Campaign' does not exist. Creating a new experiment.


Starting Model Training Pipeline...

--- XGBoost_Baseline ---
ROC-AUC:  0.7866
PR-AUC:   0.4398
Status: Logged successfully to MLflow.

--- LightGBM_Baseline ---
ROC-AUC:  0.7936
PR-AUC:   0.4487
Status: Logged successfully to MLflow.

